In [22]:
import polars as pl
import torch
import os, sys
sys.path.insert(0, os.path.abspath(".."))


MAC_DIR = '/Users/igwanhyeong/PycharmProjects/paper_research/'
WINDOW_DIR = 'C:/Users/USER/PycharmProjects/research/raw_data/'

if sys.platform == 'win32':
    DIR = WINDOW_DIR
    print(torch.cuda.is_available())
    print(torch.cuda.device_count())
    print(torch.version.cuda)
    print(torch.__version__)
    print(torch.cuda.get_device_name(0))
    print(torch.__version__)
else:
    DIR = MAC_DIR

/Users/igwanhyeong/PycharmProjects/paper_research/simple_lab_test


In [11]:
select_schema = [
    'tpep_pickup_datetime', 'tpep_dropoff_datetime',
    'pickup_longitude', 'pickup_latitude',
    'dropoff_longitude', 'dropoff_latitude',
]
rename_schema = {
    'tpep_pickup_datetime': 'pick_dt', 'tpep_dropoff_datetime': 'drop_dt',
    'pickup_longitude': 'pick_lon', 'pickup_latitude': 'pick_lat',
    'dropoff_longitude': 'drop_lon', 'dropoff_latitude': 'drop_lat'
}

vendor1_data = (
    pl.read_parquet(DIR + 'sample_data/yellow_trip.parquet')
      .filter(pl.col('VendorID') == 1) # VendorID = 1 | 2
      .select(select_schema)
      .rename(rename_schema)
 )

vendor2_data = (
    pl.read_parquet(DIR + 'sample_data/yellow_trip.parquet')
      .filter(pl.col('VendorID') == 2)
      .select(select_schema)
      .rename(rename_schema)
)

In [12]:
vendor1_data

pick_dt,drop_dt,pick_lon,pick_lat,drop_lon,drop_lat
str,str,f64,f64,f64,f64
"""2015-01-10 20:33:38""","""2015-01-10 20:53:28""",-74.001648,40.724243,-73.994415,40.759109
"""2015-01-10 20:33:38""","""2015-01-10 20:43:41""",-73.963341,40.802788,-73.95182,40.824413
"""2015-01-10 20:33:39""","""2015-01-10 20:35:31""",-74.009087,40.713818,-74.004326,40.719986
"""2015-01-10 20:33:39""","""2015-01-10 20:52:58""",-73.971176,40.762428,-74.004181,40.742653
"""2015-01-10 20:33:39""","""2015-01-10 20:53:52""",-73.874374,40.774048,-73.986977,40.758194
…,…,…,…,…,…
"""2015-01-10 19:01:44""","""2015-01-10 19:05:40""",-73.951988,40.786217,-73.953735,40.775162
"""2015-01-10 19:01:44""","""2015-01-10 19:07:26""",-73.982742,40.728184,-73.974976,40.720013
"""2015-01-10 19:01:44""","""2015-01-10 19:15:01""",-73.979324,40.74955,-73.969101,40.7878


# 목표: pick_dt 기반 “도착 과정”을 Homogeneous Poisson Process로 모델링/검증

## 개념 정리
- (Homogeneous) Poisson Process:
  - 일정한 rate λ(초당/분당/시간당)로 이벤트가 발생
  - 성질:
    1) Inter-arrival time(이벤트 간격) ~ Exponential(λ)
    2) 구간 길이 Δ에 대한 카운트 N(Δ) ~ Poisson(λΔ)
    3) Disjoint 구간의 increments 독립 (실무에서는 근사적으로 bin 카운트 상관이 낮은지 확인)

## 데이터에서 무엇을 이벤트로 볼지
- 이벤트 시각: pick_dt (pickup 발생 시간)
- 공간 조건(선택): 특정 지역(bounding box 또는 grid cell)만 필터링하면 “거의 homogeneous”에 가까운 구간을 만들기 쉬움

### 1) Polars Datetime 파싱 및 기본 전처리

In [16]:
def preprocess(df: pl.DataFrame) -> pl.DataFrame:
    df = (df
            .with_columns([
                pl.col('pick_dt').str.strptime(pl.Datetime, format = '%Y-%m-%d %H:%M:%S', strict = False).alias('pick_ts'),
                pl.col('drop_dt').str.strptime(pl.Datetime, format = '%Y-%m-%d %H:%M:%S', strict = False).alias('drop_ts'),
            ])
            .drop(['pick_dt', 'drop_dt'])
    )

    df = df.filter(pl.col('pick_ts').is_not_null())
    return df

### 2) Temporal Poisson Process Modeling (Homogeneous)

2.1 특정 시간 구간 + 특정 공간 필터

Poisson 가정이 성립하려면, rate가 크게 변하지 않는 구간을 잡는 것이 중요.

    Example)
        - 특정 날짜 하루
        - 특정 시간대 (Ex: 20:00 ~ 21:00)
        - 특정 Location bounding box

In [18]:
from datetime import datetime
def filter_window(df: pl.DataFrame, start: str, end: str, bbox = None) -> pl.DataFrame:
    start_dt = datetime.strptime(start, '%Y-%m-%d %H:%M:%S')
    end_dt = datetime.strptime(end, '%Y-%m-%d %H:%M:%S')

    out = df.filter(
        (pl.col('pick_ts') >= pl.lit(start_dt)) &
        (pl.col('pick_ts') < pl.lit(end_dt))
    )

    if bbox is not None:
        min_lon, min_lat, max_lon, max_lat = bbox
        out = out.filter(
            (pl.col('pick_lon') >= min_lon) & (pl.col('pick_lon') <= max_lon) &
            (pl.col('pick_lat') >= min_lat) & (pl.col('pick_lat') <= max_lat)
        )
    return out

2.2 λ(레이트) 추정 (MLE)

관측 구간 길이를 T(초 단위)라 하면, Homogeneous Poisson process의 MLE는 $\hat{\lambda} = \frac{N}{T}$

In [24]:
def estimate_lambda(df_win: pl.DataFrame) -> float:
    # pick_ts ascending
    ts = df_win.sort(pl.col('pick_ts')).sort('pick_ts')
    n = ts.height
    if n < 2:
        raise ValueError('이벤트가 너무 작음(최소 2개 이상 권장).')

    t0 = ts[0, 0]
    t1 = ts[-1, 0]

    T_seconds = float(t1 - t0)
    if T_seconds <= 0:
        raise ValueError('시간 범위 T가 0 이하')
    lam = n / T_seconds
    return lam

## 3) Poisson Process Validation

### Check A) Inter-arrival time이 Exponential(λ) 인가? </br>
Poisson Process이면, 이벤트 간격 $\Delta t_i = t_i - t_{i-1}$ 가 i.i.d. Exp(λ) 입니다. </br>
아래는 KS test + 기본 통계(평균≈1/λ, 분산≈1/λ²) 체크입니다.

In [25]:
import numpy as np
def get_interarrival_seconds(df_win: pl.DataFrame) -> np.ndarray:
    ts = df_win.select(pl.col('pick_ts')).sort('pick_ts').to_series()
    t = ts.to_list()
    dt = np.array([(t[i] - t[i - 1]).total_seconds() for i in range(1, len(t))], dtype = float)
    return dt

def check_exponential(dt: np.ndarray, lam: float):
    # base diagnosis
    mean_dt = dt.mean()
    var_dt = dt.var(ddof = 1)

    # 이론 값
    theo_mean = 1.0 / lam
    theo_var = 1.0 / (lam ** 2)

    out = {
        'emp_mean_dt': mean_dt,
        'emp_var_dt': var_dt,
        'theo_mean_dt': theo_mean,
        'theo_var_dt': theo_var,
    }
    return out

from scipy import stats
def ks_test_exponential(dt: np.ndarray, lam: float):
    # Exp(scale=1/lam)
    # scipy의 expon은 scale 파라미터 사용
    D, p = stats.kstest(dt, 'expon', args = (0, 1.0/lam))
    return {'KS_D': float(D), 'p_value': float(p)}

### Check B) Bin-count가 Poisson(λΔ)인가? (Mean≈Var 확인 포함)</br>
관측 구간을 동일 길이 bin(예: 1분/5분/10분)으로 나누고, 각 bin의 이벤트 수를 세면:</br>
$N_j \sim \text{Poisson}(\lambda \Delta)$

In [26]:
def bin_counts(df_win: pl.DataFrame, bin_size_sec: int = 60) -> np.ndarray:
    tmp = df_win.select([(pl.col('pick_ts').dt.epoch('s')).alias('tsec')]).sort('tsec')
    t0 = tmp[0, 0]
    tmp = tmp.with_columns(
        ((pl.col('tsec') - t0) // bin_size_sec).cast(pl.Int64).alias('bin')
    )
    counts = tmp.group_by('bin').len().sort('bin')['len'].to_numpy()
    return counts

def check_poisson_counts(counts: np.ndarray, lam: float, bin_size_sec: int):
    emp_mean = counts.mean()
    emp_var = counts.var(ddof = 1)
    theo_mean = lam * bin_size_sec

    # Poisson이면 mean≈var≈λΔ
    return {
        'emp_mean': float(emp_mean),
        'emp_var': float(emp_var),
        'theo_mean_lambdaDelta': float(theo_mean),
        'overdispersion_var_over_mean': float(emp_var / emp_mean) if emp_mean > 0 else None
    }

In [27]:
def simulate_poisson_process(lam: float, T_seconds: float, seed: int = 0) -> np.ndarray:
    rng = np.random.default_rng(seed)
    t = 0.0
    times = []
    while True:
        # inter-arrival ~ Exp(lam)
        dt = rng.exponential(scale=1.0/lam)
        t += dt
        if t > T_seconds:
            break
        times.append(t)
    return np.array(times, dtype=float)  # event times in seconds from 0

In [ ]:
import polars as pl

# 예: parquet/csv 로딩 (원하는 방식 선택)
# df = pl.read_parquet("your_data.parquet")
# df = pl.read_csv("your_data.csv")

df = preprocess(vendor1_data)

# 예시: 하루 구간 + (선택) bbox
df_win = filter_window(
    df,
    start="2015-01-10 20:00:00",
    end="2015-01-10 21:00:00",
    bbox=None  # (min_lon, min_lat, max_lon, max_lat) 넣으면 공간 제한
)

lam = estimate_lambda(df_win)
dt = get_interarrival_seconds(df_win)

print("lambda_hat (events/sec):", lam)
print("lambda_hat (events/min):", lam*60)

print("Exponential check:", check_exponential(dt, lam))
print("KS test:", ks_test_exponential(dt, lam))

counts = bin_counts(df_win, bin_size_sec=60)
print("Poisson count check:", check_poisson_counts(counts, lam, bin_size_sec=60))

# 시뮬레이션 비교
ts_sorted = df_win.select("pick_ts").sort("pick_ts").to_series().to_list()
T_seconds = (ts_sorted[-1] - ts_sorted[0]).total_seconds()
sim_times = simulate_poisson_process(lam, T_seconds, seed=42)

print("Observed N:", len(ts_sorted))
print("Simulated N:", len(sim_times))

lambda_hat (events/sec):  3484472.5194805195
lambda_hat (events/min):  209068351.16883117
Exponential check:  {'emp_mean_dt': 0.2511864879955332, 'emp_var_dt': 0.19717874151630319, 'theo_mean_dt': 2.8698748358878846e-07, 'theo_var_dt': 8.236181573662513e-14}
KS test:  {'KS_D': 0.7533500837520938, 'p_value': 0.0}
Poisson count check:  {'emp_mean': 238.81666666666666, 'emp_var': 544.6946327683615, 'theo_mean_lambdaDelta': 209068351.16883117, 'overdispersion_var_over_mean': 2.2808066135879472}


In [23]:
from models.poisson_process import WindowSpec, HPPAnalyzer, PiecewiseNHPPAnalyzer

# df = pl.read_parquet("your_data.parquet")  # 또는 CSV
# 아래는 사용자가 가진 df를 그대로 넣는 형태를 가정
df = vendor1_data

spec = WindowSpec(
    start="2015-01-10 20:00:00",
    end="2015-01-10 21:00:00",
    bbox=None,
)

# HPP
hpp = HPPAnalyzer(df)
hpp_result = hpp.run(spec, bin_size_sec=60, sim_seed=42)
print("HPP result keys:", hpp_result.keys())
print("HPP lambda_hat (events/min):", hpp_result["lambda_hat_events_per_min"])
print("HPP exp KS:", hpp_result["diagnose_exponential"]["ks"])
print("HPP bin overdispersion:", hpp_result["diagnose_bin_counts"]["overdispersion_var_over_mean"])

# NHPP (piecewise)
nhpp = PiecewiseNHPPAnalyzer(df)
nhpp_result = nhpp.run(spec, bin_size_sec=300, sim_seed=42)
print("NHPP result keys:", nhpp_result.keys())
print("NHPP rescaling KS:", nhpp_result["diagnostics"].get("ks_exp"))
print("NHPP bins:", nhpp_result["fit"]["B"])

HPP result keys: dict_keys(['model', 'window', 'lambda_hat_events_per_sec', 'lambda_hat_events_per_min', 'n_events', 'span_seconds_first_to_last', 'diagnose_exponential', 'bin_size_sec', 'bin_counts', 'diagnose_bin_counts', 'simulated_times_sec_from0', 'simulated_n'])
HPP lambda_hat (events/min): 188.86913031397611
HPP exp KS: {'KS_D': 0.6959745762711864, 'p_value': 0.0}
HPP bin overdispersion: 1.9142398913243501
NHPP result keys: dict_keys(['model', 'window', 'bin_size_sec', 'n_events', 'fit', 'time_rescaling_z', 'diagnostics', 'simulated_times_sec_epoch', 'simulated_n'])
NHPP rescaling KS: {'KS_D': 0.6959745762711864, 'p_value': 0.0}
NHPP bins: 12
